# Spark DataFrames API

The majority of the data that we will deal with in this course is either structured or semistructured. The PySpark DataFrame API is a higher-level abstraction over PySpark Core for processing these datasets.

The APIs provided by PySpark are highly optimized. PySpark can read data from many file types such as CSV files, JSON files, and files from other databases. We work with **DataFrames**, which are similar to tables in a relational database or DataFrames in pandas/polars. The DataFrame consists of named columns and is a collection of ``Row`` objects.

In this notebook, we will explore the core functionalities of the Spark DataFrames API, compare it to other popular data manipulation libraries, configure our Spark cluster, and complete some exercises using a great dataset – the New York Squirrel Census.

## Comparing Spark, Pandas, and Polars

When introducing Spark DataFrames, comparing them to single-node libraries like Pandas and Polars is an excellent way to anchor the new material to what you might already know.

### 1. The Execution Model: Lazy vs. Eager Evaluation
This is arguably the most important conceptual shift for users coming from Pandas.
* **Pandas (Eager):** When you run a line of code, Pandas executes it immediately. If you filter a 10 GB dataset and then select one column, Pandas does the heavy lifting right then and there.
* **Spark (Lazy):** Spark builds a Directed Acyclic Graph (DAG) of your transformations. It doesn't actually compute anything until you call an "action" (like `.show()`, `.count()`, or `.write()`). This allows Spark's Catalyst Optimizer to find the most efficient execution plan before moving a single byte of data. 
* **Polars (Lazy/Eager):** Polars supports both. Its eager API feels like Pandas, but its lazy API (using `.lazy()` and `.collect()`) operates much like Spark's optimizer to push down predicates and minimize memory usage.

### 2. Architecture: Distributed vs. Single-Node
Understanding where the computation happens prevents fundamental design mistakes.
* **Spark:** Distributed by design. A Spark DataFrame looks like one table, but under the hood, it is chunked into "partitions" distributed across multiple worker nodes in a cluster. 
* **Pandas:** Single-node and single-threaded. It operates strictly within the RAM of the machine running the script.
* **Polars:** Single-node but heavily multi-threaded. It takes full advantage of all available CPU cores on a single machine, often outperforming Spark on datasets that fit on a single computer.

### 3. Immutability
In Spark, DataFrames are entirely immutable.
* **Spark / Polars:** You cannot change a DataFrame "in place." Every transformation creates a brand new DataFrame reference. This immutability is what allows Spark to safely distribute tasks and recover from node failures.
* **Pandas:** Allows in-place mutations (e.g., `df.drop('col', inplace=True)`), which can lead to memory overhead or unpredictable states if not managed carefully.

### 4. Memory Management and Out-of-Core Processing
What happens when the dataset is larger than your RAM?
* **Pandas:** Throws an `Out of Memory` (OOM) error and crashes.
* **Spark:** Designed to handle petabytes of data. If data doesn't fit in RAM, Spark gracefully spills the excess to disk, continuing the computation (albeit slower).
* **Polars:** Uses a streaming engine for its lazy API to process data out-of-core, meaning it can handle datasets larger than RAM by streaming batches through the CPU, much like Spark.

### 5. Syntax and API Translation

| Operation | Pandas | Polars | PySpark |
| :--- | :--- | :--- | :--- |
| **Select Columns** | `df[['A', 'B']]` | `df.select(['A', 'B'])` | `df.select('A', 'B')` |
| **Filter Rows** | `df[df['A'] > 5]` | `df.filter(pl.col('A') > 5)` | `df.filter(col('A') > 5)` |
| **Create/Modify Column** | `df['C'] = df['A'] * 2` | `df.with_columns((pl.col('A') * 2).alias('C'))` | `df.withColumn('C', col('A') * 2)` |
| **Group and Count** | `df.groupby('A').size()` | `df.group_by('A').len()` | `df.groupBy('A').count()` |
| **Sort / Order** | `df.sort_values('A')` | `df.sort('A')` | `df.orderBy('A')` |
| **Null Fill** | `df.fillna(0)` | `df.fill_null(0)` | `df.fillna(0)` |


## Setup and Cluster Configuration

First, we need to set up our environment by installing the required dependencies. (If you set them up already, you can skip the dependency installation.)

Then, we will initialize our Spark session, taking a moment to configure how much memory and processing power our Spark cluster is allowed to use.

A Spark application consists of a **Driver** program (which runs your main script and creates the SparkSession) and **Executor** nodes (which do the actual distributed data processing).

When building a `SparkSession`, you can specify hardware limits:
* **`.master("local[4]")`**: Tells Spark to run locally using 4 CPU cores. Using `local[*]` tells it to use all available logical cores on your machine.
* **`spark.executor.memory`**: Controls how much RAM is allocated to each executor node for processing data.
* **`spark.driver.memory`**: Controls how much RAM is allocated to the driver node (useful if you are calling `.collect()` or `.toPandas()` to bring large amounts of data back to the driver).

The following cell is important to run nonetheless, since it is required for Spark's native plotting capabilities.

In [8]:
!pip install pyspark plotly "pandas>=2.2.0" "nbformat>=4.2.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 113.0 MB/s  0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 2.1.4
    Uninstalling pandas-2.1.4:
      Successfully uninstalled pandas-2.1.4


In [ ]:
# Install Java 17 (Required for Spark)
!sudo apt-get update
!sudo apt-get install -y openjdk-17-jdk-headless
!java -version

In [1]:
# Set JAVA_HOME and initialize Spark Session with specific configurations
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

from pyspark.sql import SparkSession

spark = SparkSession.builder \
        .master("local[4]") \
        .appName("PySpark DataFrames API") \
        .config("spark.executor.memory", "4g") \
        .config("spark.driver.memory", "2g") \
        .getOrCreate()

print("Spark Session configured and ready!")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/13 18:41:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark Session configured and ready!


## Part 1: Loading Data and Core Verbs (Subsetting & Filtering)

Data manipulation is often conceptualized as a series of "verbs" or actions you take on a dataset. Let's load the `housing.csv` dataset and start with the most basic verbs: subsetting and filtering.

* **Column References (`col`)**: Before applying operations, you often need to reference specific columns. The `col()` function selects a column as a specific DataFrame column object, which is necessary for evaluating logical conditions (like `col("area") > 5000`).
* **Subsetting (`select`)**: Allows you to pick specific columns from a potentially massive DataFrame, reducing the amount of data carried in memory.
* **Filtering (`filter` or `where`)**: Restricts rows based on Boolean conditions.

In [2]:
from pyspark.sql.functions import col

# Load housing data into a dataframe
df = spark.read.csv('housing.csv', header=True, inferSchema=True)

# Select specific columns and show the top 5 rows
print("--- Subsetting Columns ---")
df.select("price", "area", "bedrooms", "furnishingstatus").show(5)

# Filter the dataframe based on multiple conditions
print("--- Filtering Data (Area > 5000 AND Bedrooms >= 3) ---")
df.filter((col("area") > 5000) & (col("bedrooms") >= 3)) \
  .select("price", "area", "bedrooms", "furnishingstatus") \
  .show(5)

--- Subsetting Columns ---
+--------+----+--------+----------------+
|   price|area|bedrooms|furnishingstatus|
+--------+----+--------+----------------+
|13300000|7420|       4|       furnished|
|12250000|8960|       4|       furnished|
|12250000|9960|       3|  semi-furnished|
|12215000|7500|       4|       furnished|
|11410000|7420|       4|       furnished|
+--------+----+--------+----------------+
only showing top 5 rows
--- Filtering Data (Area > 5000 AND Bedrooms >= 3) ---
+--------+----+--------+----------------+
|   price|area|bedrooms|furnishingstatus|
+--------+----+--------+----------------+
|13300000|7420|       4|       furnished|
|12250000|8960|       4|       furnished|
|12250000|9960|       3|  semi-furnished|
|12215000|7500|       4|       furnished|
|11410000|7420|       4|       furnished|
+--------+----+--------+----------------+
only showing top 5 rows


## Part 2: Mutating, Renaming, and Sorting

Often, the data you need doesn't exist yet; you have to derive it from what you have. 

* **Mutating (`withColumn`)**: This verb creates a new column (or overwrites an existing one) by applying a transformation. 
* **Renaming (`withColumnRenamed`)**: Simply changes the name of a column without altering its data.
* **Sorting (`orderBy` or `sort`)**: Reorders the rows of your DataFrame based on one or more columns.

In [3]:
from pyspark.sql.functions import desc

df_enhanced = df.withColumn("price_per_sqft", col("price") / col("area")) \
                .withColumnRenamed("furnishingstatus", "furnishing_status")

# Sort to find the top 5 most expensive houses per square foot
df_enhanced.select("price", "area", "price_per_sqft", "furnishing_status") \
           .orderBy(desc("price_per_sqft")) \
           .show(5)

+-------+----+------------------+-----------------+
|  price|area|    price_per_sqft|furnishing_status|
+-------+----+------------------+-----------------+
|9240000|3500|            2640.0|        furnished|
|4340000|1905| 2278.215223097113|   semi-furnished|
|8750000|4320| 2025.462962962963|   semi-furnished|
|4270000|2175|1963.2183908045977|      unfurnished|
|4200000|2145| 1958.041958041958|      unfurnished|
+-------+----+------------------+-----------------+
only showing top 5 rows


## Part 3: Cleaning Data

Cleaning data is arguably the most common task in a data pipeline. Spark provides several methods to handle missing, duplicated, or malformed data.

* **Dropping Columns (`drop`)**: Removes columns entirely from the DataFrame.
* **Dropping Duplicates (`dropDuplicates` or `distinct`)**: Removes duplicate rows based on all or a subset of columns.
* **Dropping Nulls (`dropna`)**: Removes any row that contains a null value. You can use the `subset` parameter to only check specific columns.
* **Filling Nulls (`fillna`)**: Replaces null values with a specified default value (can be passed as a dictionary mapping columns to specific defaults).
* **Conditional Logic (`when().otherwise()`)**: Similar to an IF/ELSE statement, frequently used inside `withColumn` to correct anomalous values based on conditions.

In [4]:
from pyspark.sql.functions import col, when

# Let's create a messy DataFrame to demonstrate cleaning operations
messy_data = [
    (1, "A", None, "Keep"),
    (2, None, 50, "DropMe"),
    (3, "C", 100, "Keep"),
    (3, "C", 100, "Keep")  # Exact duplicate row
]
df_messy = spark.createDataFrame(messy_data, ["id", "category", "value", "unwanted_col"])

df_messy.show()

# 1. Drop a specific column
df_clean1 = df_messy.drop("unwanted_col")

# 2. Drop duplicates
df_clean2 = df_clean1.dropDuplicates()

# 3. Handle Nulls (dropna on a subset, and fillna for the rest)
# We will drop rows where 'category' is null, but fill 'value' nulls with 0
df_clean3 = df_clean2.dropna(subset=["category"]) \
                     .fillna({"value": 0})

# 4. Conditional Replacement (Fixing bad data)
# If category is "A", let's replace it with "Alpha"
df_final = df_clean3.withColumn("category",
    when(col("category") == "A", "Alpha").otherwise(col("category"))
)

df_final.show()

+---+--------+-----+------------+
| id|category|value|unwanted_col|
+---+--------+-----+------------+
|  1|       A| NULL|        Keep|
|  2|    NULL|   50|      DropMe|
|  3|       C|  100|        Keep|
|  3|       C|  100|        Keep|
+---+--------+-----+------------+

+---+--------+-----+
| id|category|value|
+---+--------+-----+
|  3|       C|  100|
|  1|   Alpha|    0|
+---+--------+-----+



## Part 4: Grouping and Aggregation

To extract patterns and summarize data, we perform aggregations.
* **Grouping (`groupBy`)**: Partitions the DataFrame into distinct groups based on one or more columns. This returns a `GroupedData` object.
* **Simple Aggregations**: You can directly call functions like `.count()`, `.mean()`, `.max()`, or `.min()` on the grouped data.
* **Multiple Aggregations (`agg`)**: If you need to calculate several different statistics simultaneously, you use the `.agg()` method. Inside `.agg()`, you can apply functions like `sum`, `countDistinct`, and `round`, and rename the outputs using `.alias()`.
* **Chaining**: You can seamlessly chain `.orderBy()` after aggregations to sort the summarized results.

In [5]:
from pyspark.sql.functions import max, min, avg, sum, countDistinct, round, desc

# 1. Simple Grouping
df.groupBy('furnishingstatus').count().show()

# 2. Grouping by multiple columns
df.groupBy('bedrooms', 'furnishingstatus') \
  .mean('area') \
  .orderBy('bedrooms', desc('furnishingstatus')) \
  .show()

# 3. Comprehensive aggregations using .agg(), .alias(), and .round()
df.groupBy('furnishingstatus').agg(
    min('price').alias('cheapest_price'),
    max('price').alias('highest_price'),
    round(avg('price'), 2).alias('average_price'),
    sum('area').alias('total_area_sqft'),
    countDistinct('bedrooms').alias('unique_bedroom_counts')
).show()

+----------------+-----+
|furnishingstatus|count|
+----------------+-----+
|  semi-furnished|  227|
|     unfurnished|  178|
|       furnished|  140|
+----------------+-----+

+--------+----------------+-----------------+
|bedrooms|furnishingstatus|        avg(area)|
+--------+----------------+-----------------+
|       1|     unfurnished|           3970.0|
|       1|       furnished|           3450.0|
|       2|     unfurnished|4288.912280701755|
|       2|  semi-furnished|          4675.14|
|       2|       furnished|5251.827586206897|
|       3|     unfurnished|4814.350515463918|
|       3|  semi-furnished|5227.692913385827|
|       3|       furnished|5751.013157894737|
|       4|     unfurnished|4808.444444444444|
|       4|  semi-furnished|5580.833333333333|
|       4|       furnished|6064.275862068966|
|       5|     unfurnished|           8092.5|
|       5|  semi-furnished|           3602.5|
|       5|       furnished|           5835.0|
|       6|     unfurnished|           3600

## Part 5: Visualisation

PySpark DataFrames come with built-in plotting capabilities, primarily powered by the `plotly` backend. Because Spark is designed for massive datasets, the plotting API usually aggregates or samples the data behind the scenes before sending it to the browser to render, preventing out-of-memory errors on your local machine.

Let's explore some basic plots using our housing dataset. We will thoroughly comment the parameters so you understand exactly how to build rich, multi-dimensional charts.

In [6]:
# A simple scatter plot to visualize the relationship between Area and Price.
# We pass the exact column names as strings to the x and y parameters.
df.plot.scatter(x="area", y="price")

In [7]:
# Box plots are great for visualizing the distribution of a single variable and identifying outliers.
# Here, we first subset the DataFrame to just the "area" column before calling .plot.box()
df.select("area").plot.box()

We can also create much more informative, multi-dimensional plots by leveraging additional parameters like color and size.

In [8]:
# Creating a rich scatter plot:
# - x="price" and y="area": Defines our standard 2D axes.
# - size="bedrooms": Adds a 3rd dimension. The physical size of the bubble corresponds to the number of bedrooms.
# - color="parking": Adds a 4th dimension. The color of the bubble represents whether the house has parking.
# - hover_name="parking": When you interactively hover over a bubble with your mouse, a tooltip will boldly display the parking status.
# - color_continuous_scale="Plasma": Changes the default color palette to 'Plasma' for better visual distinction.
fig = df.plot.scatter(
    x="price",
    y="area",
    size="bedrooms",
    hover_name="parking",
    color="parking",  
    color_continuous_scale="Plasma",  
)

# Render the interactive plot
fig.show()

## Part 6: Joins

Often we are required to combine information from two or more DataFrames based on a common key. PySpark supports standard SQL join types:
* **Inner Join**: Keeps only rows with matching keys in both DataFrames.
* **Left Outer Join**: Keeps all rows from the left DataFrame, filling in nulls if there is no match in the right DataFrame.
* **Right & Full Outer Joins**: Works similarly to left joins, but keeping right or all records, respectively.

If the columns you want to join on have different names in the left and right DataFrames, you can specify the condition explicitly (e.g., `col('key1') == col('key2')`).

You can also join on multiple columns (a composite key) by passing a list of column names, or by combining multiple conditions using the `&` operator.

In [9]:
from pyspark.sql.functions import col

# Let's create two small DataFrames manually
authors_data = [
    (1, 'Isaac Asimov'), 
    (2, 'Frank Herbert'), 
    (3, 'Arthur C. Clarke'), 
    (4, 'George Orwell')
]
df_authors = spark.createDataFrame(authors_data, ['author_id', 'author_name'])
 
books_data = [
    (101, 'Foundation', 1), 
    (102, 'Dune', 2), 
    (103, '1984', 4), 
    (104, 'The Hobbit', 5)
]
df_books = spark.createDataFrame(books_data, ['book_id', 'book_name', 'writer_id'])

# Perform an inner join where the keys have different names
df_inner = df_authors.join(df_books, col('author_id') == col('writer_id'), 'inner')
df_inner.show()

# Perform a left outer join on the same different keys
df_left = df_authors.join(df_books, col('author_id') == col('writer_id'), 'left')
df_left.show()

# Perform a right outer join to see books without authors in our author table
df_right = df_authors.join(df_books, col('author_id') == col('writer_id'), 'right')
df_right.show()

# --- Composite Key Join Example ---

# Let's create another dataset in the authors domain to demonstrate joining on multiple keys
events_data = [
    (1, '2026-05-01', 'NYC Book Expo'), 
    (1, '2026-06-15', 'London Book Fair'), 
    (2, '2026-05-01', 'Sci-Fi Convention')
]
df_events = spark.createDataFrame(events_data, ['author_id', 'event_date', 'event_name'])

signings_data = [
    (1, '2026-05-01', 500), 
    (1, '2026-06-15', 300), 
    (3, '2026-05-01', 150)
]
df_signings = spark.createDataFrame(signings_data, ['author_id', 'event_date', 'books_signed'])

# Perform an inner join on both 'author_id' and 'event_date'
df_composite = df_events.join(df_signings, ['author_id', 'event_date'], 'inner')
df_composite.show()

# Note: If the composite key columns had different names (e.g. event_date vs signing_date), 
# you would combine conditions like this:
# condition = (col('author_id') == col('signing_author_id')) & (col('event_date') == col('signing_date'))
# df_events.join(df_signings, condition, 'inner')

+---------+-------------+-------+----------+---------+
|author_id|  author_name|book_id| book_name|writer_id|
+---------+-------------+-------+----------+---------+
|        1| Isaac Asimov|    101|Foundation|        1|
|        2|Frank Herbert|    102|      Dune|        2|
|        4|George Orwell|    103|      1984|        4|
+---------+-------------+-------+----------+---------+

+---------+----------------+-------+----------+---------+
|author_id|     author_name|book_id| book_name|writer_id|
+---------+----------------+-------+----------+---------+
|        1|    Isaac Asimov|    101|Foundation|        1|
|        2|   Frank Herbert|    102|      Dune|        2|
|        3|Arthur C. Clarke|   NULL|      NULL|     NULL|
|        4|   George Orwell|    103|      1984|        4|
+---------+----------------+-------+----------+---------+

+---------+-------------+-------+----------+---------+
|author_id|  author_name|book_id| book_name|writer_id|
+---------+-------------+-------+------

---

# The Great New York Squirrel Census Exercises

Now it's your turn. We are going to analyze the 2018 Central Park Squirrel Census data. Make sure you have the `squirrel_census.csv` file available in your working environment.

In [10]:
# Load the Squirrel Census data
file_path = "squirrel_census.csv"

# NOTE: Make sure squirrel_census.csv is uploaded/available in the directory before running this!
squirrel_df = spark.read.csv(file_path, header=True, inferSchema=True)
squirrel_df.printSchema()

root
 |-- X: double (nullable = true)
 |-- Y: double (nullable = true)
 |-- Unique Squirrel ID: string (nullable = true)
 |-- Hectare: string (nullable = true)
 |-- Shift: string (nullable = true)
 |-- Date: integer (nullable = true)
 |-- Hectare Squirrel Number: integer (nullable = true)
 |-- Age: string (nullable = true)
 |-- Primary Fur Color: string (nullable = true)
 |-- Highlight Fur Color: string (nullable = true)
 |-- Combination of Primary and Highlight Color: string (nullable = true)
 |-- Color notes: string (nullable = true)
 |-- Location: string (nullable = true)
 |-- Above Ground Sighter Measurement: string (nullable = true)
 |-- Specific Location: string (nullable = true)
 |-- Running: boolean (nullable = true)
 |-- Chasing: boolean (nullable = true)
 |-- Climbing: boolean (nullable = true)
 |-- Eating: boolean (nullable = true)
 |-- Foraging: boolean (nullable = true)
 |-- Other Activities: string (nullable = true)
 |-- Kuks: boolean (nullable = true)
 |-- Quaas: boolean

## Exercise 1: Filtering and Selection

The **`filter()`** (or **`where()`**) method subsets rows based on a condition (like SQL's `WHERE` clause). The **`select()`** method chooses a subset of columns. To correctly reference columns within these functions, you must use the **`col()`** function, which ensures the column name is interpreted by Spark.

**Specification:** Create a DataFrame that counts the total number of squirrels observed in the **"AM"** shift AND had **"Gray"** as their `Primary Fur Color`.
**Expected Output:** A single integer printed to the console representing the total count.

In [11]:
from pyspark.sql.functions import col

# Filter `squirrel_df` for Shift == "AM" and `Primary Fur Color` == "Gray". 
filtered_squirrels = squirrel_df.filter((col("Shift") == "AM") & (col("Primary Fur Color") == "Gray"))

# Chain the .count() method to the filtered DataFrame and print the result.
print(filtered_squirrels.count())


1087


## Exercise 2: Simple Aggregation

The **`groupBy()`** method partitions data into groups. After grouping, aggregation functions like **`count()`** summarize the data. The **`orderBy()`** method arranges the resulting rows, typically using **`desc()`** for descending order.

**Specification:** Find the three most common squirrel `Location` types.
**Expected Output:** A DataFrame containing two columns (`Location` and `count`), sorted in descending order by the count, showing only the top 3 rows.

In [12]:
from pyspark.sql.functions import desc

# Group `squirrel_df` by 'Location', apply count(), order by count descending, and show top 3
squirrel_df.groupBy("Location") \
           .count() \
           .orderBy(desc("count")) \
           .show(3)


+------------+-----+
|    Location|count|
+------------+-----+
|Ground Plane| 2115|
|Above Ground|  843|
|        NULL|   64|
+------------+-----+
only showing top 3 rows


## Exercise 3: Calculated Aggregation

The **`agg()`** method computes one or more aggregate functions (like **`avg()`** for the mean) across the data. The **`round()`** function is used to limit decimal places, ensuring clean output for numerical results.

**Specification:** Calculate the park-wide average longitude (X) and average latitude (Y) specifically for squirrels that exhibited the `Chasing` behavior (where the `Chasing` column is `true` or `TRUE`). 
**Expected Output:** A DataFrame with a single row containing two columns (representing the average X and average Y), with both values rounded to 3 decimal places.

In [13]:
from pyspark.sql.functions import avg, round, col

# Filter `squirrel_df` to keep only rows where 'Chasing' is true/TRUE
chasing_squirrels = squirrel_df.filter((col("Chasing") == "true") | (col("Chasing") == "TRUE") | (col("Chasing") == True))

# Use .agg() to calculate the avg() of 'X' and 'Y', rounded to 3 decimal places
chasing_squirrels.agg(
    round(avg("X"), 3).alias("avg_X"),
    round(avg("Y"), 3).alias("avg_Y")
).show()


+-------+-----+
|  avg_X|avg_Y|
+-------+-----+
|-73.968|40.78|
+-------+-----+



## Exercise 4: DateTime Processing and Visualization (`to_date()` and `.plot()`)

The **`to_date()`** function converts a date string (like the 'Date' column in this dataset, formatted as MMDDYYYY) into a proper Spark `DateType`. 

**Specification:** Analyze the sighting trend over time by converting the date and visualizing it.
**Expected Output:** An interactive area chart where the X-axis is the `ObservationDate` and the Y-axis represents the daily `count` of squirrel sightings.

In [ ]:
from pyspark.sql.functions import to_date, col

# Convert the 'Date' string (format 'MMDDYYYY') into a new DateType column called 'ObservationDate'
df_dates = squirrel_df.withColumn("ObservationDate", to_date(col("Date").cast("string"), "MMddyyyy"))

# Group by 'ObservationDate', count the sightings per day, and order chronologically
daily_counts = df_dates.groupBy("ObservationDate").count().orderBy("ObservationDate")

# Plot the resulting DataFrame using .plot.area()
fig = daily_counts.plot.area(x="ObservationDate", y="count")
fig.show()
